In [ ]:
# --timeframe 1d   : Таймфрейм свечей (дневные данные).
# --start-year 2000: Глубина загрузки истории (начиная с 2000 года).
# --workers 6      : Количество параллельных потоков для ускорения загрузки.

!python -m _tools.update_market_data --timeframe 1d --start-year 2000 --workers 6
!python -m _tools.update_macro --timeframe 1d --start-year 2000 --workers 6
# Выполняет комплексную проверку целостности, отсутствия пропусков и корректности OHLCV данных.
!python -m _tools.check_data_quality

In [3]:
# --timeframe 1d       : Интервал данных — дневные свечи.
# --lookback 60        : Глубина истории — модель смотрит на 60 дней назад.
# --horizon 10         : Горизонт прогноза — ищем выход по барьерам в течение 10 дней.
# --auto               : Режим автоматического расчета уровней TP/SL на основе волатильности.
# --percentile 75      : Перцентиль волатильности для отсечения аномальных выбросов при авто-разметке.
# --init_split         : Дата начала первого разделения данных на Train и Val.
# --val_interval 2     : Продолжительность валидационного периода в годах.
# --split_interval 2   : Шаг смещения окна Walk-Forward в годах.
# --endpoint           : Дата окончания формирования всех временных интервалов.
# --corr_threshold     : Порог удаления коррелирующих признаков (убираем дубликаты > 85%).
# --cum_threshold      : Порог кумулятивной важности (оставляем топ фичей, дающих 99% влияния).
# --force              : Раскомментируйте параметр ниже для полной перезаписи кэшированных данных.

!python -m _tools.init_dataset \
    --timeframe 1d \
    --lookback 30 \
    --horizon 5 \
    --auto \
    --percentile 75 \
    --init_split 2010-01-01 \
    --val_interval 2 \
    --split_interval 2 \
    --endpoint 2024-01-01 \
    --corr_threshold 0.85 \
    --cum_threshold 0.99 \
    #--force


🧹 Запуск модуля очистки данных (Сплиты, Иглы, Выбросы)...
Корректировка сплитов: 100%|████████████████████| 68/68 [00:00<00:00, 71.43it/s]
✅ Очистка завершена!
⏭️ [fold_2010/train - Build] Пропущен (уже существует: dataset.csv)
⏭️ [fold_2010/train - Labels] Пропущен (уже существует: labels.csv)
⏭️ [fold_2010/train - Features] Пропущен (уже существует: ml_data.parquet)
⏭️ [fold_2010/val - Build] Пропущен (уже существует: dataset.csv)
⏭️ [fold_2010/val - Labels] Пропущен (уже существует: labels.csv)
⏭️ [fold_2010/val - Features] Пропущен (уже существует: ml_data.parquet)
⏭️ [fold_2010 - Feature Selection] Пропущен (уже существует: features_selected.json)
⏭️ [fold_2010/train - TFRecords] Пропущен (уже существует: data.tfrecord)
⏭️ [fold_2010/val - TFRecords] Пропущен (уже существует: data.tfrecord)
⏭️ [fold_2012/train - Build] Пропущен (уже существует: dataset.csv)
⏭️ [fold_2012/train - Labels] Пропущен (уже существует: labels.csv)
⏭️ [fold_2012/train - Features] Пропущен (уже существует

In [ ]:
#полная проверка подготовленных на предыдущем этапе данных
!python -m _tools.verify_data

In [ ]:
!python run_walkforward.py \
    --dataset_dir "data/processed/2000_2026_1d_60_10" \
    --runs 100 \
    --batch_size 8192 \
    --epochs 100 \
    --l2_reg "1e-4" \
    --lr "1e-3" \
    --start_fold "fold_2024" \
    --append

🚀 Запуск массового обучения моделей (Walk-Forward)...
📁 Датасет: data/processed/2000_2026_1d
⚙️  Настройки: 100 runs, 100 epochs, batch 8192
⏭️ Пропускаем завершенные фолды. Начинаем строго с: fold_2024

🔥 Обучение нейросети для: fold_2024
✅ Mixed precision включена!
✅ Динамическое выделение видеопамяти включено!
🚀 Старт обучения. Фолд: [fold_2024]
📊 Форма данных: [Lookback: 60, Features: 68]
⚙️ Расчет идеальных весов классов...
   Баланс: SL(0)=44680, Hold(1)=108445, TP(2)=44497
   Веса:   SL(0)=1.47, Hold(1)=0.61, TP(2)=1.48
⏳ Подготовка конвейера данных...

--------------------------------------------------
🔄 ИТЕРАЦИЯ 1/100 (Лучшая точность сессии: 0.00%)
--------------------------------------------------
Epoch 1/100
25/25 - 11s - 439ms/step - accuracy: 0.4389 - loss: 1.2538 - val_accuracy: 0.4110 - val_loss: 1.1914
Epoch 2/100
25/25 - 6s - 226ms/step - accuracy: 0.5085 - loss: 1.0944 - val_accuracy: 0.4329 - val_loss: 1.1594
Epoch 3/100
25/25 - 6s - 220ms/step - accuracy: 0.5278 - 

In [3]:
#очистка наименне успешных ltsm моделей (остается топ 3)
!python -m _tools.clean_lstm_models

🧹 Запуск универсальной уборки моделей (Оставляем Топ-3 по val_loss)...

📂 [2000_2026_1d] Проверка фолда: fold_2010
  🏆 Оставляем:
     1. Run 40 | Loss: 0.9527 | Acc: 67.45%
     2. Run 54 | Loss: 1.0086 | Acc: 59.76%
     3. Run 14 | Loss: 1.0125 | Acc: 64.98%
  ✨ Удалять нечего, количество моделей в норме.

📂 [2000_2026_1d] Проверка фолда: fold_2012
  🏆 Оставляем:
     1. Run 5 | Loss: 1.0594 | Acc: 57.50%
     2. Run 86 | Loss: 1.0683 | Acc: 57.87%
     3. Run 1 | Loss: 1.0721 | Acc: 55.00%
  ✨ Удалять нечего, количество моделей в норме.

📂 [2000_2026_1d] Проверка фолда: fold_2014
  🏆 Оставляем:
     1. Run 20 | Loss: 1.1427 | Acc: 50.52%
     2. Run 66 | Loss: 1.1630 | Acc: 50.92%
     3. Run 1 | Loss: 1.2284 | Acc: 47.86%
  ✨ Удалять нечего, количество моделей в норме.

📂 [2000_2026_1d] Проверка фолда: fold_2016
  🏆 Оставляем:
     1. Run 5 | Loss: 1.0030 | Acc: 61.92%
     2. Run 4 | Loss: 1.0120 | Acc: 61.88%
     3. Run 1 | Loss: 1.0328 | Acc: 61.46%
  ✨ Удалять нечего, количес

In [ ]:
!python -m _tools.evaluate_lstm_predictions

In [ ]:
!python -m _tools.prepare_rl_env

In [ ]:
!python -m _tools.train_rllib_pbt --population 4 --iterations 3000